In [1]:
import sys
from pathlib import Path
root = Path.cwd().parents[1]
sys.path.append(str(root))

In [2]:
from pprint import pprint
import os
import logging

debug = False
logger = logging.getLogger()
logging.basicConfig(
    level=logging.DEBUG if debug else logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

from src.preprocessing import Reader, FixedCharChunker, Document, Page, Embedder
from src.indexing import DataStore, ChunkStore, FlatVectorStore, Index
from src.utils import AutoEDAIndex
from src.llm import OpenAILLM

import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

base_url = os.getenv('BASE_OPEN_AI_URL')
api_key = os.getenv("OPEN_AI_KEY")
client = OpenAI(api_key=api_key, base_url=base_url)

/Users/danilaandreev/Documents/HSE/degree/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Инициализация индекса

In [3]:
DATA_STORE_DIR = '../../data/index/documents'
CHUNK_STORE_DIR = '../../data/index/chunks'
VECTOR_STORE_DIR = '../../data/index/vectors'
SQLITE_DIR = '../../data/index/db'

embedder = Embedder(
    device='mps', 
    # model_name='ai-forever/FRIDA',
    # type_handlers={
    #     'query': lambda text: f'search_query: {text}',
    #     'document': lambda text: f'search_document: {text}',
    # }
)
dimensions = embedder.get_embeddings(['test']).shape[-1]

index = Index(
    datastore=DataStore(DATA_STORE_DIR, logger),
    vectorstore=FlatVectorStore(VECTOR_STORE_DIR, dimensions, logger),
    chunkstore=ChunkStore(CHUNK_STORE_DIR, logger),
    chunker=FixedCharChunker(chunk_size=500, overlap=50, logger=logger),
    embedder=embedder,
    reader=Reader(logger),
    sqlite_path=SQLITE_DIR,
    logger=logger
)

2026-03-31 20:52:21,283 | INFO | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: sergeyzh/LaBSE-ru-turbo
2026-03-31 20:52:21,678 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sergeyzh/LaBSE-ru-turbo/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-31 20:52:21,747 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sergeyzh/LaBSE-ru-turbo/055975b9638dd075bec75500321f57bc36cb1f4d/modules.json "HTTP/1.1 200 OK"
2026-03-31 20:52:21,918 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sergeyzh/LaBSE-ru-turbo/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-03-31 20:52:22,091 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sergeyzh/LaBSE-ru-turbo/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-03-31 20:52:22,092 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Pl

In [4]:
# index.clear()
index.info()

2026-03-31 20:45:19,611 | INFO | root | Index stats: {'datastore': {'documents': 175}, 'chunkstore': {'chunks': 58762}, 'vectorstore': {'vectors': 58980, 'dimensions': 768}, 'index': {'points': 58762, 'points_without_chunk': 0, 'chunks_without_point': 0}}


{'datastore': {'documents': 175},
 'chunkstore': {'chunks': 58762},
 'vectorstore': {'vectors': 58980, 'dimensions': 768},
 'index': {'points': 58762,
  'points_without_chunk': 0,
  'chunks_without_point': 0}}

## Загрузка документов

In [6]:
SOURCE_DATA_DIR = '../../data/raw_data'

sources = [os.path.join(SOURCE_DATA_DIR, name) for name in os.listdir(SOURCE_DATA_DIR)]
print(f'Количество источников: {len(sources)}')

doc_ids = index.add_sources(sources)
index.save_vectorstore()

Indexing ocr_test.pdf | RSS 3063 MB | MPS 490/4071 MB:  85%|████████▍ | 148/175 [09:08<01:44,  3.86s/it]  2026-03-01 17:09:52,786 | INFO | root | File 8dfd9c2a58fc3a913f4f73f7f82b59ff9d21544c4b6da5b7e3b7a42558369527.json already exists
Indexing 4294850881.pdf | RSS 3171 MB | MPS 490/3304 MB:  90%|█████████ | 158/175 [09:45<01:05,  3.87s/it]2026-03-01 17:10:29,959 | INFO | root | File 3a87da2057da5f074af2abfb0b0833578bf341e10b7696543bcb511439e05e5d.json already exists
2026-03-01 17:10:29,960 | INFO | root | File 02d7943892e1a20f6f425e0ea644e41bd583f8f60d4085543c6a75dc092e0206.json already exists
2026-03-01 17:10:29,960 | INFO | root | File 8bc1ef122417226c49ced7b7f6f05317bb772caa9706df61464a536039e81f30.json already exists
2026-03-01 17:10:29,960 | INFO | root | File 1ec27b5f90b408a680c0e0e053aa7476527ceb75402837226ce4182873ae565d.json already exists
2026-03-01 17:10:29,960 | INFO | root | File 1b33947dfb90a921ac1dc7f0e5260dcf110bef9453ffdc5211cde2b4e9db46e8.json already exists
2026-03-

In [8]:
index.info()

2026-03-01 17:11:47,202 | INFO | root | Index stats: {'datastore': {'documents': 175}, 'chunkstore': {'chunks': 58762}, 'vectorstore': {'vectors': 58980, 'dimensions': 768}, 'index': {'points': 58762, 'points_without_chunk': 0, 'chunks_without_point': 0}}


{'datastore': {'documents': 175},
 'chunkstore': {'chunks': 58762},
 'vectorstore': {'vectors': 58980, 'dimensions': 768},
 'index': {'points': 58762,
  'points_without_chunk': 0,
  'chunks_without_point': 0}}

In [9]:
test_query = 'Нагрузка перекрытий'
res = index.search(test_query)

for r in res:
    print(r.chunk['text'])
    print()

тажным пере­ крытием. Перекрытие состоит из железобетон­ ной несущей плиты у = 2500 кг/м3 толщиной 10 см, звукоизоляционных полосовых прокла­ док из жестких минераловатных плит плотнос­ тью 140 кг/м3 толщиной 4 см в необжатом со­ стоянии и дощатого пола толщиной 35 мм на лагах сечением 100x50 мм с шагом 50 см. По­ лезная нагрузка 2000 Па. Определяем поверхностные плотности эле­ ментов перекрытия: /и, = 2500 • 0,1 = 250 кг/м2; т2 = 600 • 0,035 (доски) + 600 0,05 0,1-2 (лаги) = 27 кг/м2. Нагрузка 

адии проектирования данные об этих условиях недостаточны, при расчете конструкций и оснований необходимо рассмотреть следующие варианты загружения отдельных перекрытий: сплошное загружение принятой нагрузкой; неблагоприятное частичное загружение при расчете конструкций и оснований, чувствительных к такой схеме загружения; отсутствие временной нагрузки. 8.1 Определение нагрузок от оборудования, складируемых материалов и изделий 8.1.1 Нагрузки от оборудования (в том числе трубопроводов, транспор

# Baseline

In [6]:
eda = AutoEDAIndex(index)
df = eda.run()
# df

DOCUMENTS
- count: 175
- text_length_mean: 140687.2857142857
- text_length_median: 108903.0
- text_length_min: 1874
- text_length_max: 693714

CHUNKS
- count: 58762
- text_length_mean: 457.77097443926345
- text_length_median: 500.0
- text_length_min: 51
- text_length_max: 500
- text_length_std: 104.0179440315079
- words_mean: 63.612691875701984
- sentences_mean: 5.673564548517749
- paragraphs_mean: 1.0

EMBEDDINGS
- matrix_rows: 58980
- matrix_cols: 768
- dtype: float64
- dimension_std_mean: 0.020093295640934438
- dimension_std_min: 0.011243343291884037
- dimension_std_max: 0.030654873981914122
- pairwise_similarity_pairs: 1739290710
- pairwise_similarity_min: 0.2903698521986882
- pairwise_similarity_max: 1.0000002282914577
- pairwise_similarity_mean: 0.6858390089736611
- pairwise_similarity_std: 0.054904902531348145



In [11]:
from pydantic import BaseModel, Field
from typing import Optional, List
    
llm = OpenAILLM(client, model_name='gpt-5.2')

# class Answer(BaseModel):
#     answer: str
# res = llm.parse('Скажи привет', Answer)
# print(res)

In [12]:
query = 'Какие бывают нагрузки на перекрытия в многоэтажных зданиях?'
extracted = index.search(query, top_k=10)
# context = '\n\n'.join([f'{i+1}. Doc {e.chunk['doc_id']}: page {e.chunk['page_number']}\n{e.chunk['text']}' for i, e in enumerate(extracted)])
context = '\n\n'.join([f'{i+1}. {e.chunk['text']}' for i, e in enumerate(extracted)])
prompt = f'Ты отвечаешь на вопросы пользователя по строительной документации. Отвечай только на основе доступного контекста.\n\nЗапрос: {query}\n\nКонтекст: {context}'
class Answer(BaseModel):
    reasoning: List[str] = Field(..., description='3-5 предложений по обдумыванию контекста')
    answer: str = Field(..., description='Финальный ответ')
answer = llm.parse(prompt, Answer)
pprint(answer.answer)

2026-03-31 23:19:24,553 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"


('По приведённому контексту на перекрытия в многоэтажных зданиях '
 'рассматриваются следующие нагрузки/воздействия:\n'
 '\n'
 '- Временные эксплуатационные нагрузки: от людей, животных, оборудования, '
 'изделий, материалов, временных перегородок, транспортных средств (для '
 'перекрытий, покрытий, лестниц и полов по грунту).\n'
 '- Сосредоточенные вертикальные нагрузки (в неблагоприятном положении на '
 'площадке до 10×10 см). Если не задано иное: \n'
 '  - для перекрытий и лестниц — 1,5 кН;\n'
 '  - для чердачных перекрытий, покрытий, террас и балконов — 1,0 кН;\n'
 '  - для покрытий, где передвижение только по трапам/мостикам — 0,5 кН.\n'
 '- Дополнительные нагрузки и воздействия при реконструкции (их нужно '
 'учитывать при проверке несущих/ограждающих конструкций и грунтов '
 'основания).\n'
 '- Воздействия, связанные с обеспечением комфортности проживания: параметры '
 'колебаний перекрытий верхних этажей.\n'
 '- Горизонтальные нагрузки (упоминается передача горизонтальной нагру

После создания бенчмарка требуется сделать таблицу, которая будет показывать все основные метрики качества при различных комбинациях создания индекса. 
- Сначала сделаем бейзлайн
- Потом попробуем перебрать разные варианты чанкирования - выберем лучший для по метрикам
- Потом переберем разные варианты моделей - выберем лучшие
- Потом будет пробовать изменить архитектуру - выбираем лучшую
- Потом допиливаем через дообучение моделей
Всё должно собираться в dataframe для сохранения результатов